In [ ]:
!pip install diffusers transformers accelerate ftfy
!pip install git+https://github.com/tencent-ailab/IP-Adapter.git

### **ip-adapter base code**

In [ ]:
import torch
from diffusers import AutoPipelineForText2Image
from diffusers.utils import load_image
import requests
from io import BytesIO
from PIL import Image
import numpy as np

# Load the IP-Adapter pipeline (uses SDXL base with IP-Adapter and T5 for multimodal prompts)
pipeline = AutoPipelineForText2Image.from_pretrained(
    "stabilityai/stable-diffusion-xl-base-1.0",
    torch_dtype=torch.float16,
    variant="fp16"
).to("cuda")

pipeline.load_ip_adapter(
    "h94/IP-Adapter",
    subfolder="models",
    weight_name="ip-adapter_sdxl.bin"
)

# Load CLIP vision for image encoding
from transformers import CLIPVisionModelWithProjection
clip_vision = CLIPVisionModelWithProjection.from_pretrained(
    "openai/clip-vision-large-patch14"
).to("cuda", torch.float16)
# No projection needed for IP-Adapter; use the vision model directly

# Load T5 for text encoding (multimodal prompt support)
from transformers import T5EncoderModel, T5Tokenizer
t5_encoder = T5EncoderModel.from_pretrained("google/t5-v1_1-xxl").to("cuda", torch.float16)
t5_tokenizer = T5Tokenizer.from_pretrained("google/t5-v1_1-xxl")

# Function to encode image with CLIP vision
def encode_image(image):
    image = np.array(image)
    if image.ndim == 2:
        image = image[:, :, None] * np.array([0.48145466, 0.4578275, 0.40821073])
    elif image.ndim == 3 and image.shape[-1] == 4:
        image = image[:, :, :3]
    image = image.astype(np.float32) / 255.0
    image = torch.from_numpy(image).permute(2, 0, 1).unsqueeze(0).to("cuda", torch.float16)
    image = torch.nn.functional.interpolate(
        image, size=(224, 224), mode="bilinear", align_corners=False
    )
    with torch.no_grad():
        image_features = clip_vision(image).image_embeds
    return image_features

# Function to encode text with T5
def encode_text_with_t5(prompt):
    inputs = t5_tokenizer(prompt, padding=True, truncation=True, return_tensors="pt")
    inputs = {k: v.to("cuda") for k, v in inputs.items()}
    with torch.no_grad():
        encoder_outputs = t5_encoder(**inputs)
    return encoder_outputs.last_hidden_state

# Input image (replace with your image path or URL)
image_url_or_path = 'train\images\\aral_387.jpg'  # e.g., 'https://example.com/image.jpg' or '/content/your_image.jpg'

if image_url_or_path.startswith('http'):
    response = requests.get(image_url_or_path)
    image = Image.open(BytesIO(response.content)).convert("RGB")
else:
    image = Image.open(image_url_or_path).convert("RGB")

# Your text prompt (replace this)
prompt = "without changing the location of object make this image realistic"  # e.g., "a photo of a person wearing a red dress"

# Encode inputs
image_embeds = encode_image(image)
text_embeds = encode_text_with_t5(prompt)

# Prepare multimodal embeds for IP-Adapter (concatenate or average as per demo; here we use simple concat for demo)
# In the full demo, this is handled via custom attention; this approximates for basic use
combined_embeds = torch.cat([image_embeds, text_embeds.mean(dim=1)], dim=-1)  # Simple combo; adjust if needed

# Set IP-Adapter scale (0.0 to 1.0; higher = more image influence)
pipeline.set_ip_adapter_scale(0.8)

# Generate image
negative_prompt = ""  # Optional: e.g., "blurry, low quality"
generator = torch.Generator(device="cuda").manual_seed(42)  # For reproducibility
output_image = pipeline(
    prompt=prompt,
    image_embeds=image_embeds,  # Pass image embeds
    text_embeds=text_embeds,    # Pass T5 text embeds
    negative_prompt=negative_prompt,
    num_inference_steps=28,
    generator=generator,
    width=1024,
    height=1024
).images[0]

# Save and display
output_image.save("generated_image.png")
output_image.show()  # In Colab, use output_image to display

### **simple**

In [ ]:
import torch
from diffusers import AutoPipelineForText2Image
from diffusers.utils import load_image
from PIL import Image
import requests
from io import BytesIO

pipeline = AutoPipelineForText2Image.from_pretrained(
    "stabilityai/stable-diffusion-xl-base-1.0",
    torch_dtype=torch.float16,
    variant="fp16",
    use_safetensors=True
).to("cuda")

pipeline.load_ip_adapter(
    "h94/IP-Adapter",
    subfolder="sdxl_models",
    weight_name="ip-adapter_sdxl.bin"
)

pipeline.set_ip_adapter_scale(0.8)

image_path_or_url = '0 content image.png'  

if image_path_or_url.startswith('http'):
    response = requests.get(image_path_or_url)
    image = Image.open(BytesIO(response.content)).convert("RGB")
else:
    image = Image.open(image_path_or_url).convert("RGB")


Loading pipeline components...: 100%|██████████| 7/7 [00:00<00:00, 10.51it/s]


In [ ]:

prompt = "Image style transfer. Use Image as the base. Isolate the background region and recolor and re-texture it to match the aesthetic, lighting, and color gradient of the background, add more detial to image." 

negative_prompt = "blurry, low quality, deformed"

generator = torch.Generator(device="cuda").manual_seed(42) 
output_image = pipeline(
    prompt=prompt,
    ip_adapter_image=image,
    negative_prompt=negative_prompt,
    num_inference_steps=50,
    generator=generator,
).images[0]

output_image.save("generated_image_2.png")
output_image.show()  

100%|██████████| 50/50 [00:31<00:00,  1.57it/s]
